In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

np.random.seed(42)

# File paths

SPLIT_DIR = Path("../backend/data/splits_70_15_15")
OUT_DIR = Path("../backend/data/Enkf_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = SPLIT_DIR / "train.parquet"
VAL_PATH = SPLIT_DIR / "val.parquet"
TEST_PATH = SPLIT_DIR / "test.parquet"

# Output tag 
MODEL_TAG = "EnKF"

# Configuration

OBS_COLS = ["temperature", "humidity", "audio_density"]
D = len(OBS_COLS)
H = np.eye(D)

MIN_ROWS_VAL = 200
MIN_ROWS_TEST = 200

# Candidate process-noise scales to tune on the validation split
Q_SCALES = [0.03, 0.1, 0.3, 1.0, 3.0, 10.0]

# Ensemble Kalman Filter parameters
N_ENS = 60
INFLATE = 1.05
SEED = 42

# Initial state covariance (state uncertainty, not measurement noise)
P0 = np.eye(D) * 10.0  # matches the KF baseline P0

# Process model selection:
# - For fairness relative to the KF baseline with A = I, use a random-walk model by default.
USE_AR1 = False
PHI = 0.995  # used only if USE_AR1 is True

# Time-gap scaling for process noise Q
BASE_DT_MIN = 15.0
USE_DT_AWARE_Q = True
DT_SCALE_MODE = "since_observed"  # options: "row_dt" or "since_observed"
DT_CLIP_MIN = 1.0
DT_CLIP_MAX = 16.0

# Missing data behavior: increase process uncertainty when no observations are available
MISSING_Q_SCALE = 2.0

# Optional value clipping bounds computed from TRAIN only (disabled by default)
CLIP_QUANTILES = None
CLIP_AFTER_UPDATE_ONLY = True

# Numerical stability for NIS computation
MIN_JITTER = 1e-8


# Load and clean data

def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize types, remove invalid rows, and sort by hive/time.
    Ensures:
      - published_at is UTC datetime
      - tag_number is integer-like
      - observation columns are numeric
    """
    df = df.copy()
    df["published_at"] = pd.to_datetime(df["published_at"], utc=True, errors="coerce")
    df["tag_number"] = pd.to_numeric(df["tag_number"], errors="coerce").astype("Int64")
    for c in OBS_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["published_at", "tag_number"])
    return df.sort_values(["tag_number", "published_at"]).reset_index(drop=True)

print("Loading data splits...")
train_df = clean_df(pd.read_parquet(TRAIN_PATH))
val_df = clean_df(pd.read_parquet(VAL_PATH))
test_df = clean_df(pd.read_parquet(TEST_PATH))
print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)

# Utility functions

def rmse_per_dim(y, yhat):
    """Compute RMSE per dimension, ignoring NaNs."""
    y = np.asarray(y, float)
    yhat = np.asarray(yhat, float)
    out = np.full((y.shape[1],), np.nan, float)
    for d in range(y.shape[1]):
        m = np.isfinite(y[:, d]) & np.isfinite(yhat[:, d])
        if m.sum() > 0:
            out[d] = float(np.sqrt(np.mean((y[m, d] - yhat[m, d]) ** 2)))
    return out

def persistence_forecast_missing_safe(z):
    """
    Baseline forecast: persistence using the last observed value per dimension.
    Produces NaN until the first observation arrives for that dimension.
    """
    T, d = z.shape
    out = np.full((T, d), np.nan, float)
    last = np.full((d,), np.nan, float)
    for t in range(T):
        out[t] = last
        m = np.isfinite(z[t])
        last[m] = z[t, m]
    return out

def init_x0(z):
    """Initialize the state using the first available observation in each dimension."""
    x0 = np.zeros(D, float)
    for t in range(len(z)):
        m = np.isfinite(z[t])
        if m.any():
            x0[m] = z[t, m]
            break
    return x0

def usable_rows_df(df_h: pd.DataFrame) -> int:
    """Count rows with at least one finite observation."""
    return int(np.isfinite(df_h[OBS_COLS].to_numpy(float)).any(axis=1).sum())

# TRAIN-only normalization for aggregated RMSE (prevents leakage)
train_std = train_df[OBS_COLS].astype(float).std(skipna=True).to_numpy()
train_std = np.where(np.isfinite(train_std) & (train_std > 1e-12), train_std, 1.0)

def agg_zrmse(rmse_raw):
    """Aggregate per-dimension RMSE after normalization by TRAIN standard deviation."""
    rmse_raw = np.asarray(rmse_raw, float)
    m = np.isfinite(rmse_raw) & np.isfinite(train_std) & (train_std > 1e-12)
    if m.sum() == 0:
        return np.nan
    return float(np.mean(rmse_raw[m] / train_std[m]))

def percentile_threshold(arr, p=95.0, min_n=200):
    """Return the p-th percentile threshold if there are at least min_n finite samples."""
    a = np.asarray(arr, float)
    a = a[np.isfinite(a)]
    if a.size < min_n:
        return np.nan
    return float(np.percentile(a, p))


# TRAIN-only R estimate (robust-ish, simple)

def estimate_R_from_train(df: pd.DataFrame) -> np.ndarray:
    """
    Estimate measurement noise covariance R using TRAIN only.
    Uses per-hive first-difference variance as a proxy and aggregates via the median.
    """
    vals = []
    for _, g in df.groupby("tag_number"):
        z = g[OBS_COLS].to_numpy(float)
        if len(z) < 6:
            continue
        vals.append(0.5 * np.nanvar(np.diff(z, axis=0), axis=0))
    if len(vals) == 0:
        return np.diag([1e-3] * D)
    R_diag = np.nanmedian(np.vstack(vals), axis=0)
    R_diag = np.where(np.isfinite(R_diag) & (R_diag > 1e-8), R_diag, 1e-3)
    R_diag = np.clip(R_diag, 1e-6, None)
    return np.diag(R_diag)

R = estimate_R_from_train(train_df)
R_diag = np.diag(R)
print("R diag (TRAIN-only):", R_diag)


# Optional clipping bounds (TRAIN-only)

if CLIP_QUANTILES is not None:
    q_lo, q_hi = CLIP_QUANTILES
    q = train_df[OBS_COLS].quantile([q_lo, q_hi])
    LOW = q.loc[q_lo].to_numpy(float)
    HIGH = q.loc[q_hi].to_numpy(float)
else:
    LOW = None
    HIGH = None


# dt scaling utilities (time-gap aware)

def dt_row_minutes(ts: np.ndarray) -> np.ndarray:
    """Compute per-row time delta in minutes (based on consecutive timestamps)."""
    ts = pd.to_datetime(ts, utc=True, errors="coerce").to_numpy()
    dt = np.full(len(ts), np.nan, float)
    if len(ts) >= 2:
        d = (ts[1:] - ts[:-1]) / np.timedelta64(1, "m")
        dt[1:] = d.astype(float)
    dt[0] = BASE_DT_MIN
    dt = np.where(np.isfinite(dt) & (dt > 0), dt, BASE_DT_MIN)
    return dt

def dt_since_last_observed_minutes(df_h: pd.DataFrame) -> np.ndarray:
    """
    Compute minutes since the most recent row that had ANY observed value.
    This makes the process noise scale increase over missing stretches.
    """
    has_any = df_h[OBS_COLS].notna().any(axis=1).to_numpy()
    ts = pd.to_datetime(df_h["published_at"], utc=True, errors="coerce").to_numpy()

    out = np.full(len(df_h), np.nan, float)
    last_t = None
    for i in range(len(df_h)):
        if last_t is None:
            out[i] = BASE_DT_MIN
        else:
            out[i] = (ts[i] - last_t) / np.timedelta64(1, "m")
        if has_any[i]:
            last_t = ts[i]

    out = np.where(np.isfinite(out) & (out > 0), out, BASE_DT_MIN)
    return out

def build_dt_scale(df_h: pd.DataFrame) -> np.ndarray | None:
    """
    Build a per-timestep scaling factor for Q based on time gaps.
    The scaling is linear in dt relative to BASE_DT_MIN and clipped to [DT_CLIP_MIN, DT_CLIP_MAX].
    """
    if not USE_DT_AWARE_Q:
        return None

    if DT_SCALE_MODE == "row_dt":
        dt = dt_row_minutes(df_h["published_at"].to_numpy())
    elif DT_SCALE_MODE == "since_observed":
        dt = dt_since_last_observed_minutes(df_h)
    else:
        raise ValueError(f"Unknown DT_SCALE_MODE: {DT_SCALE_MODE}")

    s = dt / BASE_DT_MIN
    s = np.where(np.isfinite(s) & (s > 0), s, 1.0)
    return np.clip(s, DT_CLIP_MIN, DT_CLIP_MAX)


# Chi-square quantiles (for classical NIS consistency checks)

def _chi2_ppf_wilson_hilferty(p: float, k: int) -> float:
    """Approximate chi-square inverse CDF using Wilson–Hilferty transform."""
    try:
        from scipy.stats import norm
        z = float(norm.ppf(p))
    except Exception:
        # Acklam inverse normal approximation
        def inv_norm_cdf(pp):
            a = [-3.969683028665376e+01, 2.209460984245205e+02,
                 -2.759285104469687e+02, 1.383577518672690e+02,
                 -3.066479806614716e+01, 2.506628277459239e+00]
            b = [-5.447609879822406e+01, 1.615858368580409e+02,
                 -1.556989798598866e+02, 6.680131188771972e+01,
                 -1.328068155288572e+01]
            c = [-7.784894002430293e-03, -3.223964580411365e-01,
                 -2.400758277161838e+00, -2.549732539343734e+00,
                 4.374664141464968e+00, 2.938163982698783e+00]
            d = [7.784695709041462e-03, 3.224671290700398e-01,
                 2.445134137142996e+00, 3.754408661907416e+00]
            plow = 0.02425
            phigh = 1 - plow
            if pp < plow:
                q = np.sqrt(-2 * np.log(pp))
                return (((((c[0] * q + c[1]) * q + c[2]) * q + c[3]) * q + c[4]) * q + c[5]) / \
                       ((((d[0] * q + d[1]) * q + d[2]) * q + d[3]) * q + 1)
            if pp > phigh:
                q = np.sqrt(-2 * np.log(1 - pp))
                return -(((((c[0] * q + c[1]) * q + c[2]) * q + c[3]) * q + c[4]) * q + c[5]) / \
                        ((((d[0] * q + d[1]) * q + d[2]) * q + d[3]) * q + 1)
            q = pp - 0.5
            r = q * q
            return (((((a[0] * r + a[1]) * r + a[2]) * r + a[3]) * r + a[4]) * r + a[5]) * q / \
                   (((((b[0] * r + b[1]) * r + b[2]) * r + b[3]) * r + b[4]) * r + 1)

        z = float(inv_norm_cdf(p))

    k = max(int(k), 1)
    return float(k * (1 - 2 / (9 * k) + z * np.sqrt(2 / (9 * k))) ** 3)

def chi2_ppf(p: float, k: int) -> float:
    """Prefer scipy.stats.chi2.ppf; otherwise use Wilson–Hilferty approximation."""
    try:
        from scipy.stats import chi2
        return float(chi2.ppf(p, df=int(k)))
    except Exception:
        return _chi2_ppf_wilson_hilferty(p, k)

def map_chi2_threshold(dof: int, p: float) -> float:
    """Return chi-square threshold for a given DOF and probability p."""
    if dof <= 0:
        return np.nan
    return chi2_ppf(p, dof)


# EnKF FAIR run (fixed P0, dt scaling, optional AR(1))

def enkf_run_fair(
    z: np.ndarray,
    Q_base: np.ndarray,
    R: np.ndarray,
    x0: np.ndarray,
    P0: np.ndarray,
    dt_scale: np.ndarray | None,
    n_ens=N_ENS,
    inflate=INFLATE,
    use_ar1=USE_AR1,
    phi=PHI,
    seed=SEED,
    missing_q_scale=MISSING_Q_SCALE,
    min_jitter=MIN_JITTER,
):
    """
    Ensemble Kalman Filter with a "fair" forecast evaluation:
      - x_pred[t] is the forecast BEFORE assimilating z[t].
      - The update step assimilates only observed dimensions at time t.
    """
    T = len(z)
    rng = np.random.default_rng(seed)

    # Initialize ensemble from the state uncertainty P0 (not from measurement noise R)
    X = rng.multivariate_normal(x0, P0, size=n_ens)

    if (LOW is not None) and (HIGH is not None):
        X = np.clip(X, LOW[None, :], HIGH[None, :])

    x_pred = np.zeros((T, D), float)
    x_filt = np.zeros((T, D), float)
    pred_std = np.zeros((T, D), float)

    nis = np.full(T, np.nan, float)
    nis_norm = np.full(T, np.nan, float)
    nis_dof = np.zeros(T, int)

    for t in range(T):
        zt = z[t]
        m = np.isfinite(zt)
        m_dim = int(m.sum())
        nis_dof[t] = m_dim

        # Scale Q linearly with dt factor (or gap-aware factor).
        scale = 1.0 if dt_scale is None else float(dt_scale[t])
        if m_dim == 0:
            scale *= missing_q_scale
        Q_eff = Q_base * scale

        # Forecast step 
        W = rng.multivariate_normal(np.zeros(D), Q_eff, size=n_ens)

        if use_ar1:
            # Optional AR(1) dynamics around the ensemble mean
            mu = X.mean(axis=0)
            X = mu + phi * (X - mu) + W
        else:
            # Random walk dynamics (aligned with KF baseline A = I)
            X = X + W

        # Apply covariance inflation
        xb = X.mean(axis=0)
        X = xb + inflate * (X - xb)

        if (not CLIP_AFTER_UPDATE_ONLY) and (LOW is not None) and (HIGH is not None):
            X = np.clip(X, LOW[None, :], HIGH[None, :])

        # Save forecast mean and spread (std)
        xb = X.mean(axis=0)
        x_pred[t] = xb
        pred_std[t] = X.std(axis=0, ddof=1)

        #  Update step 
        if m_dim > 0:
            Ht = H[m]
            Rt = R[np.ix_(m, m)]

            # Predicted measurement ensemble
            Y = X @ Ht.T
            yb = Y.mean(axis=0)
            r = zt[m] - yb

            # Innovation covariance
            if m_dim == 1:
                Pyy = np.array([[np.var(Y[:, 0], ddof=1)]], float) + Rt
            else:
                Pyy = np.cov(Y.T, bias=False) + Rt

            Pyy = 0.5 * (Pyy + Pyy.T)
            jitter = max(min_jitter, 1e-6 * (np.trace(Pyy) / max(m_dim, 1)))
            Pyy = Pyy + jitter * np.eye(m_dim)

            # NIS computed using the same Pyy used for the update
            try:
                nis[t] = float(r.T @ np.linalg.solve(Pyy, r))
            except np.linalg.LinAlgError:
                nis[t] = float(r.T @ np.linalg.pinv(Pyy) @ r)
            nis_norm[t] = nis[t] / max(m_dim, 1)

            # Cross-covariance between state and measurement
            Xc = X - xb
            Yc = Y - yb
            Pxy = (Xc.T @ Yc) / (n_ens - 1)

            # Kalman gain
            try:
                K = Pxy @ np.linalg.solve(Pyy, np.eye(m_dim))
            except np.linalg.LinAlgError:
                K = Pxy @ np.linalg.pinv(Pyy)

            # Stochastic EnKF update (perturbed observations)
            V = rng.multivariate_normal(np.zeros(m_dim), Rt, size=n_ens)
            Zp = zt[m][None, :] + V

            X = X + (Zp - Y) @ K.T

            if CLIP_AFTER_UPDATE_ONLY and (LOW is not None) and (HIGH is not None):
                X = np.clip(X, LOW[None, :], HIGH[None, :])

        x_filt[t] = X.mean(axis=0)

    return x_pred, x_filt, pred_std, nis, nis_norm, nis_dof


# Select evaluation hives from VAL only (no test-informed selection)

usable_val_hives = sorted([
    int(h) for h in val_df["tag_number"].dropna().unique()
    if usable_rows_df(val_df[val_df["tag_number"] == h]) >= MIN_ROWS_VAL
])
if len(usable_val_hives) == 0:
    raise ValueError("No usable VAL hives. Lower MIN_ROWS_VAL or verify your data splits.")

usable_test_hives = sorted([
    int(h) for h in test_df["tag_number"].dropna().unique()
    if usable_rows_df(test_df[test_df["tag_number"] == h]) >= MIN_ROWS_TEST
])

print("Usable VAL hives:", len(usable_val_hives))
print("Usable TEST hives (reporting only):", len(usable_test_hives))

pd.DataFrame({"hive_id": usable_val_hives}).to_csv(
    OUT_DIR / f"{MODEL_TAG}_eval_hives_val_only.csv", index=False
)

# Tune Q on VAL (VAL-selected hives only)

print("\nTuning Q on VAL (FAIR pre-update forecast, VAL-only hive set)...")
tuning = []
for s in Q_SCALES:
    Q = np.diag(np.clip(R_diag * float(s), 1e-8, None))
    scores = []
    for h in usable_val_hives:
        vh = val_df[val_df["tag_number"] == h]
        z = vh[OBS_COLS].to_numpy(float)

        dt_scale = build_dt_scale(vh)
        x_pred, *_ = enkf_run_fair(
            z=z,
            Q_base=Q,
            R=R,
            x0=init_x0(z),
            P0=P0,
            dt_scale=dt_scale,
            n_ens=N_ENS,
            seed=SEED,
            use_ar1=USE_AR1,
            phi=PHI,
        )
        scores.append(agg_zrmse(rmse_per_dim(z, x_pred)))

    tuning.append({
        "Q_scale": float(s),
        "median_val_agg_zrmse": float(np.nanmedian(scores)),
        "mean_val_agg_zrmse": float(np.nanmean(scores)),
        "num_hives": int(len(scores)),
    })

tuning_df = pd.DataFrame(tuning).sort_values("median_val_agg_zrmse").reset_index(drop=True)
best_scale = float(tuning_df.iloc[0]["Q_scale"])
Q_best = np.diag(np.clip(R_diag * best_scale, 1e-8, None))

tuning_df.to_csv(OUT_DIR / f"{MODEL_TAG}_q_tuning_results.csv", index=False)
print("Best Q_scale:", best_scale)
print(tuning_df)


# Run final model on VAL + TEST
# (restricted to the hive set selected from VAL, where present)

print("\nRunning final EnKF on VAL + TEST...")
rows = []
for split, df in [("val", val_df), ("test", test_df)]:
    present = set(df["tag_number"].dropna().astype(int).unique())
    run_hives = [h for h in usable_val_hives if h in present]

    for h in run_hives:
        dh = df[df["tag_number"] == h]
        if dh.empty:
            continue

        z = dh[OBS_COLS].to_numpy(float)
        dt_scale = build_dt_scale(dh)

        x_pred, x_filt, pred_std, nis, nis_norm, nis_dof = enkf_run_fair(
            z=z,
            Q_base=Q_best,
            R=R,
            x0=init_x0(z),
            P0=P0,
            dt_scale=dt_scale,
            n_ens=N_ENS,
            seed=SEED,
            use_ar1=USE_AR1,
            phi=PHI,
        )

        rows.append(pd.DataFrame({
            "published_at": pd.to_datetime(dh["published_at"].to_numpy(), utc=True),
            "hive_id": int(h),
            "split": split,

            "y_true_temperature": z[:, 0],
            "y_true_humidity": z[:, 1],
            "y_true_audio_density": z[:, 2],

            "y_pred_temperature": x_pred[:, 0],
            "y_pred_humidity": x_pred[:, 1],
            "y_pred_audio_density": x_pred[:, 2],

            "pred_std_temperature": pred_std[:, 0],
            "pred_std_humidity": pred_std[:, 1],
            "pred_std_audio_density": pred_std[:, 2],

            "enkf_filt_temperature": x_filt[:, 0],
            "enkf_filt_humidity": x_filt[:, 1],
            "enkf_filt_audio_density": x_filt[:, 2],

            "nis_raw": nis,
            "nis_norm": nis_norm,
            "nis_dof": nis_dof,

            "is_test_usable_hive": (int(h) in set(usable_test_hives)),
        }))

out = pd.concat(rows, ignore_index=True)


# VAL-only thresholds:
#  - Percentile thresholds for anomaly flagging (data-driven)
#  - Chi-square thresholds for consistency checks (theoretical)

val_mask = (out["split"] == "val") & (out["nis_dof"] > 0) & np.isfinite(out["nis_norm"])
val_scores = out.loc[val_mask, "nis_norm"].to_numpy(float)

thr95_pct = percentile_threshold(val_scores, 95.0, min_n=500)
thr99_pct = percentile_threshold(val_scores, 99.0, min_n=500)

print("\nVAL NIS_norm samples:", int(np.isfinite(val_scores).sum()))
print("Percentile anomaly thresholds (nis_norm): thr95:", thr95_pct, "thr99:", thr99_pct)

out["is_anomaly_norm_pct_p95"] = (
    (out["nis_dof"] > 0) & np.isfinite(out["nis_norm"]) & np.isfinite(thr95_pct) & (out["nis_norm"] > thr95_pct)
) if np.isfinite(thr95_pct) else np.zeros(len(out), dtype=bool)

out["is_anomaly_norm_pct_p99"] = (
    (out["nis_dof"] > 0) & np.isfinite(out["nis_norm"]) & np.isfinite(thr99_pct) & (out["nis_norm"] > thr99_pct)
) if np.isfinite(thr99_pct) else np.zeros(len(out), dtype=bool)

# Classical NIS consistency flags: nis_raw > chi2_ppf(p, dof)
out["chi2_thr_raw_p95"] = out["nis_dof"].apply(lambda k: map_chi2_threshold(int(k), 0.95))
out["chi2_thr_raw_p99"] = out["nis_dof"].apply(lambda k: map_chi2_threshold(int(k), 0.99))

out["is_inconsistent_chi2_p95"] = (
    (out["nis_dof"] > 0) & np.isfinite(out["nis_raw"]) & np.isfinite(out["chi2_thr_raw_p95"]) &
    (out["nis_raw"] > out["chi2_thr_raw_p95"])
)

out["is_inconsistent_chi2_p99"] = (
    (out["nis_dof"] > 0) & np.isfinite(out["nis_raw"]) & np.isfinite(out["chi2_thr_raw_p99"]) &
    (out["nis_raw"] > out["chi2_thr_raw_p99"])
)

# Save per-step outputs

out.to_parquet(OUT_DIR / f"{MODEL_TAG}_val_test_forecast_nis.parquet", index=False)
out[out["split"] == "val"].to_parquet(OUT_DIR / f"{MODEL_TAG}_forecasts_val.parquet", index=False)
out[out["split"] == "test"].to_parquet(OUT_DIR / f"{MODEL_TAG}_forecasts_test.parquet", index=False)


# Summary metrics (per hive)

summary_rows = []
calib_rows = []

for split_name in ["val", "test"]:
    df_split = out[out["split"] == split_name].copy()

    for hive_id, g in df_split.groupby("hive_id"):
        z = g[["y_true_temperature", "y_true_humidity", "y_true_audio_density"]].to_numpy(float)
        pred = g[["y_pred_temperature", "y_pred_humidity", "y_pred_audio_density"]].to_numpy(float)
        base = persistence_forecast_missing_safe(z)

        rmse_base = rmse_per_dim(z, base)
        rmse_model = rmse_per_dim(z, pred)

        summary_rows.append({
            "hive_id": int(hive_id),
            "split": split_name,
            "baseline_agg_zrmse": agg_zrmse(rmse_base),
            "enkf_agg_zrmse": agg_zrmse(rmse_model),
            "baseline_rmse_temp": rmse_base[0],
            "baseline_rmse_humid": rmse_base[1],
            "baseline_rmse_audio": rmse_base[2],
            "enkf_rmse_temp": rmse_model[0],
            "enkf_rmse_humid": rmse_model[1],
            "enkf_rmse_audio": rmse_model[2],
            "n_rows": int(len(g)),
            "is_test_usable_hive": bool(g["is_test_usable_hive"].iloc[0]),
        })

        # Calibration exceedance rates (chi-square based)
        m95 = (g["nis_dof"] > 0) & np.isfinite(g["nis_raw"]) & np.isfinite(g["chi2_thr_raw_p95"])
        m99 = (g["nis_dof"] > 0) & np.isfinite(g["nis_raw"]) & np.isfinite(g["chi2_thr_raw_p99"])
        calib_rows.append({
            "hive_id": int(hive_id),
            "split": split_name,
            "chi2_exceed_rate_p95": float(np.mean(g.loc[m95, "nis_raw"] > g.loc[m95, "chi2_thr_raw_p95"])) if m95.any() else np.nan,
            "chi2_exceed_rate_p99": float(np.mean(g.loc[m99, "nis_raw"] > g.loc[m99, "chi2_thr_raw_p99"])) if m99.any() else np.nan,
            "n_dof_pos": int((g["nis_dof"] > 0).sum()),
            "is_test_usable_hive": bool(g["is_test_usable_hive"].iloc[0]),
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT_DIR / f"{MODEL_TAG}_summary_metrics_per_hive.csv", index=False)

calib_df = pd.DataFrame(calib_rows)
calib_df.to_csv(OUT_DIR / f"{MODEL_TAG}_nis_chi2_calibration_per_hive.csv", index=False)

# Overall calibration summary (per split)
overall_calib = []
for split_name in ["val", "test"]:
    g = out[out["split"] == split_name]
    m95 = (g["nis_dof"] > 0) & np.isfinite(g["nis_raw"]) & np.isfinite(g["chi2_thr_raw_p95"])
    m99 = (g["nis_dof"] > 0) & np.isfinite(g["nis_raw"]) & np.isfinite(g["chi2_thr_raw_p99"])
    overall_calib.append({
        "split": split_name,
        "chi2_exceed_rate_p95_overall": float(np.mean(g.loc[m95, "nis_raw"] > g.loc[m95, "chi2_thr_raw_p95"])) if m95.any() else np.nan,
        "chi2_exceed_rate_p99_overall": float(np.mean(g.loc[m99, "nis_raw"] > g.loc[m99, "chi2_thr_raw_p99"])) if m99.any() else np.nan,
        "n_samples_dof_pos": int((g["nis_dof"] > 0).sum()),
    })
pd.DataFrame(overall_calib).to_csv(OUT_DIR / f"{MODEL_TAG}_nis_chi2_calibration_overall.csv", index=False)


# Run configuration (for reproducibility)

pd.DataFrame([{
    "MODEL_TAG": MODEL_TAG,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "EVAL_MODE": "FAIR_pre_update_forecast",
    "R_SOURCE": "TRAIN_only",
    "Q_TUNED_ON": "VAL_only (VAL-selected hives; no TEST-informed selection)",
    "VAL_HIVE_SELECTION": f"usable_rows>=MIN_ROWS_VAL ({MIN_ROWS_VAL})",
    "TEST_USABLE_CRITERIA_REPORTING_ONLY": f"usable_rows>=MIN_ROWS_TEST ({MIN_ROWS_TEST})",

    "best_Q_scale": best_scale,
    "n_ens": N_ENS,
    "inflate": INFLATE,

    "P0_diag": ",".join([f"{x:.6g}" for x in np.diag(P0)]),
    "USE_AR1": USE_AR1,
    "phi_if_ar1": PHI,

    "USE_DT_AWARE_Q": USE_DT_AWARE_Q,
    "DT_SCALE_MODE": DT_SCALE_MODE,
    "DT_CLIP_MIN": DT_CLIP_MIN,
    "DT_CLIP_MAX": DT_CLIP_MAX,
    "missing_q_scale": MISSING_Q_SCALE,

    "clip_quantiles": str(CLIP_QUANTILES),
    "clip_after_update_only": CLIP_AFTER_UPDATE_ONLY,

    "R_diag": ",".join([f"{x:.6g}" for x in R_diag]),

    # Percentile anomaly thresholds (data-driven)
    "nis_norm_pct_p95_val": thr95_pct,
    "nis_norm_pct_p99_val": thr99_pct,

    "eval_hives_val_count": len(usable_val_hives),
    "seed": SEED,
}]).to_csv(OUT_DIR / f"{MODEL_TAG}_run_config.csv", index=False)

print("\nSaved EnKF FAIR outputs to:", OUT_DIR)
print(" -", OUT_DIR / f"{MODEL_TAG}_eval_hives_val_only.csv")
print(" -", OUT_DIR / f"{MODEL_TAG}_q_tuning_results.csv")
print(" -", OUT_DIR / f"{MODEL_TAG}_val_test_forecast_nis.parquet")
print(" -", OUT_DIR / f"{MODEL_TAG}_summary_metrics_per_hive.csv")
print(" -", OUT_DIR / f"{MODEL_TAG}_nis_chi2_calibration_per_hive.csv")
print(" -", OUT_DIR / f"{MODEL_TAG}_nis_chi2_calibration_overall.csv")
print(" -", OUT_DIR / f"{MODEL_TAG}_run_config.csv")

Loading data splits...
Train: (560733, 31) Val: (120160, 31) Test: (120187, 31)
R diag (TRAIN-only): [0.0124549  0.29751825 2.82167299]
Usable VAL hives: 33
Usable TEST hives (reporting only): 27

Tuning Q on VAL (FAIR pre-update forecast, VAL-only hive set)...
Best Q_scale: 3.0
   Q_scale  median_val_agg_zrmse  mean_val_agg_zrmse  num_hives
0     3.00              0.170516            0.187143         33
1     1.00              0.170787            0.182586         33
2     0.30              0.184838            0.191330         33
3    10.00              0.188659            0.206286         33
4     0.10              0.210727            0.213402         33
5     0.03              0.256023            0.252013         33

Running final EnKF on VAL + TEST...

VAL NIS_norm samples: 57425
Percentile anomaly thresholds (nis_norm): thr95: 2.555972809398196 thr99: 7.505957301358759

Saved EnKF FAIR outputs to: ..\backend\data\Enkf_outputs
 - ..\backend\data\Enkf_outputs\EnKF_eval_hives_val_only